In [31]:
import os
from dotenv import load_dotenv
load_dotenv() 

True

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient


# Use your preferred embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect to Qdrant cloud
client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)


In [33]:
# Load your structured otic.txt
with open("otic.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split into chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_text(raw_text)


In [ ]:
# Step 5: Delete existing collection (if it exists)
# collection_name = "otic-knowledge-base"

# if client.collection_exists(collection_name):
#     client.delete_collection(collection_name)
#     print(f"🗑️ Old collection '{collection_name}' deleted.")
# else:
#     print(f"✅ No existing collection named '{collection_name}' found.")


🗑️ Old collection 'otic-knowledge-base' deleted.


In [35]:
from langchain.vectorstores import Qdrant

qdrant = Qdrant.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="otic-knowledge-base",
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)


C:\Users\JOHN PAUL\AppData\Roaming\Python\Python313\site-packages\torch\nn\modules\module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [41]:
from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash-latest",  # or "gemini-pro" if accessible
    temperature=0.1
)

retriever = qdrant.as_retriever(search_kwargs={"k": 6})


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)


In [45]:
query = "who is the CEO of Otic Foundation?"
response = qa_chain.invoke({"query": query})
print("✅ Answer:", response["result"])


# Optional: Show the chunks used to generate the answer
for i, doc in enumerate(response["source_documents"]):
    print(f"\n📄 Source Chunk {i+1}:\n{doc.page_content}")



C:\Users\JOHN PAUL\AppData\Roaming\Python\Python313\site-packages\torch\nn\modules\module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ Answer: Mr. Paul Nesta Katende is the Founder & CEO of Otic Foundation.

📄 Source Chunk 1:
---

## Team Testimonials

- **Elijah B, Operations Lead:** Working at Otic Foundation offers invaluable exposure and purpose.
- **Nakawalya Patricia, Operations Officer:** Transformed mindset and work ethic.
- **Martin Ayebazibwe, Director - Admin & Ops:** Witnessing collective impact and empowerment.

---

## Contact Information

- **Call Us:** +256 756 722263 / +256 706 867547  
- **Email Us:** info@oticfoundation.org 
- **Office Hours:** 8:00 AM - 5:00 PM 
- **Visit Us:** National ICT Innovation Hub, Nakawa, Uganda  

---

## Summary

📄 Source Chunk 2:
---

## Summary

Otic Foundation is building an inclusive, AI-ready society through education, advocacy, and innovation. Our mission, programs, and people are aligned toward national digital empowerment goals, with a clear focus on youth, equity, and sustainable development.


## Conclusion

Otic Foundation stands at the forefront of digital 